In [4]:
import numpy as np
from collections import defaultdict

def read_corpus(file_name):
    with open(file_name, "r") as file:
        words = file.read().splitlines()
    return words

def preprocess(words):
    cleaned = []
    for word in words:
        word = word.lower().strip()
        if word:
            cleaned.append(word)
    return cleaned

def get_vocab(words):
    vocab = defaultdict(int)
    for word in words:
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] += 1
    return vocab

def get_pairs(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]

    return new_vocab

def encode(word, merge_rules):
    tokens = list(word) + ["</w>"]

    for pair in merge_rules:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                tokens[i] = tokens[i] + tokens[i + 1]
                del tokens[i + 1]
            else:
                i += 1

    return tokens

def decode(tokens):
    word = "".join(tokens)
    return word.replace("</w>", "")

corpus = read_corpus("corpus.txt")
corpus = preprocess(corpus)

print("=" * 50)
print("Corpus")
print("=" * 50)
print(corpus)

vocab = get_vocab(corpus)

print("\nInitial Vocabulary\n")
for word, freq in vocab.items():
    print(word, ":", freq)

num_merges = 10
merge_rules = []

print("\n" + "=" * 50)
print("Learning Merge Rules")
print("=" * 50)

for i in range(num_merges):
    pairs = get_pairs(vocab)

    if not pairs:
        break

    best_pair = max(pairs, key=pairs.get)

    print(f"Merge {i+1}: {best_pair}  Frequency = {pairs[best_pair]}")

    merge_rules.append(best_pair)

    vocab = merge_vocab(best_pair, vocab)

final_vocab = set()

for word in vocab:
    for token in word.split():
        final_vocab.add(token)

print("\n" + "=" * 50)
print("Learned Vocabulary")
print("=" * 50)

for token in sorted(final_vocab):
    print(token)

print("\nVocabulary Size =", len(final_vocab))

print("\n" + "=" * 50)
print("Merge Rules")
print("=" * 50)

for i, rule in enumerate(merge_rules, start=1):
    print(f"{i}. {rule}")

test_words = [
    "lower",
    "lowest",
    "newer",
    "widest",
    "low",
    "new"
]

print("\n" + "=" * 50)
print("Encoding and Decoding")
print("=" * 50)

for word in test_words:
    encoded = encode(word, merge_rules)
    decoded = decode(encoded)

    print("\nOriginal :", word)
    print("Encoded  :", encoded)
    print("Decoded  :", decoded)

token_lengths = np.array([len(token) for token in final_vocab])

print("\n" + "=" * 50)
print("NumPy Statistics")
print("=" * 50)
print("Average Token Length :", np.mean(token_lengths))
print("Maximum Token Length :", np.max(token_lengths))
print("Minimum Token Length :", np.min(token_lengths))
print("Total Vocabulary Size:", len(final_vocab))

Corpus
['low', 'lower', 'lowest', 'new', 'newer', 'widest']

Initial Vocabulary

l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

Learning Merge Rules
Merge 1: ('l', 'o')  Frequency = 3
Merge 2: ('lo', 'w')  Frequency = 3
Merge 3: ('low', 'e')  Frequency = 2
Merge 4: ('r', '</w>')  Frequency = 2
Merge 5: ('s', 't')  Frequency = 2
Merge 6: ('st', '</w>')  Frequency = 2
Merge 7: ('n', 'e')  Frequency = 2
Merge 8: ('ne', 'w')  Frequency = 2
Merge 9: ('low', '</w>')  Frequency = 1
Merge 10: ('lowe', 'r</w>')  Frequency = 1

Learned Vocabulary
</w>
d
e
i
low</w>
lowe
lower</w>
new
r</w>
st</w>
w

Vocabulary Size = 11

Merge Rules
1. ('l', 'o')
2. ('lo', 'w')
3. ('low', 'e')
4. ('r', '</w>')
5. ('s', 't')
6. ('st', '</w>')
7. ('n', 'e')
8. ('ne', 'w')
9. ('low', '</w>')
10. ('lowe', 'r</w>')

Encoding and Decoding

Original : lower
Encoded  : ['lower</w>']
Decoded  : lower

Original : lowest
Encoded  : ['lowe', 'st</w>']
Decoded  